# Aggregation and Structural Chunking Integration Test

In this notebook, we perform a controlled integration test for the aggregation and chunking. We will take a document that has already been successfully processed by the VLM (sitting in `04_md_clean`), aggregate it into a single JSON artifact, and then pass it through the `StructuralChunker`. 

Our objective is to verify that the chunks are correctly split along semantic boundaries (H1-H4) and that physical page artifacts (like high-res PNGs) are accurately mapped into the `VectorPayload` metadata for downstream Qdrant ingestion.

In [1]:
# Enable autoreload to automatically pick up changes in local modules
%load_ext autoreload
%autoreload 2

In [2]:
import logging
import json
from pathlib import Path
from IPython.display import display, Markdown

from doc_agent.configs.settings import settings
from doc_agent.utils.logger import setup_logger

# Configure logging
logger = setup_logger(
    name="008_aggregation_and_chunking_test", 
    level=logging.INFO,
    log_file="integration_test.log"
)
logger.info("Environment initialized.")

2026-05-18 17:32:32 |     INFO | 008_aggregation_and_chunking_test:3763130530.py:15 - Environment initialized.


## Step 1: Infrastructure Initialization
We bind the `StorageManager` and `ManifestManager` to a document that has already passed the extraction and healing phases.

In [3]:
from doc_agent.core.storage import LocalStorageManager
from doc_agent.core.manifest_manager import LocalManifestManager
from doc_agent.schemas.manifest import GlobalStatus

DOC_ID = "pue_1.1-1.3" 
WORKSPACE_DIR = settings.PROCESSING_DIR / DOC_ID

# 1. Initialize Core Services
storage = LocalStorageManager(base_dir=WORKSPACE_DIR)
manifest = LocalManifestManager(storage=storage, doc_id=DOC_ID)

# 2. Trigger the manifest load from disk
# We use the actual source path so the manifest resolves correctly.
source_pdf_path = settings.RAW_DIR / f"{DOC_ID}.pdf"
manifest.init_manifest(source_pdf_path=source_pdf_path, total_pages=0)

logger.info(f"Workspace initialized at: {WORKSPACE_DIR.relative_to(settings.PROJECT_ROOT)}")
logger.info(f"Manifest Manager tracking Doc ID: {manifest.doc_id}")

# 3. Validation: Rely strictly on global_status
state = manifest.state
if state and state.global_status == GlobalStatus.COMPLETED:
    logger.info(f"Manifest loaded successfully! Global Status: {state.global_status.value.upper()}")
    logger.info(f"Ready for aggregation. Total processed pages: {state.total_pages}")
else:
    current_status = state.global_status.value if state else "Empty/Not Loaded"
    logger.error(f"Cannot proceed. Manifest state is: {current_status}")

2026-05-18 17:32:32 |     INFO | 008_aggregation_and_chunking_test:504268089.py:17 - Workspace initialized at: data/02_interim/pue_1.1-1.3
2026-05-18 17:32:32 |     INFO | 008_aggregation_and_chunking_test:504268089.py:18 - Manifest Manager tracking Doc ID: pue_1.1-1.3
2026-05-18 17:32:32 |     INFO | 008_aggregation_and_chunking_test:504268089.py:23 - Manifest loaded successfully! Global Status: COMPLETED
2026-05-18 17:32:32 |     INFO | 008_aggregation_and_chunking_test:504268089.py:24 - Ready for aggregation. Total processed pages: 27


## Step 2: Document Aggregation
We use the `aggregate_document` function to connect the individual cleaned Markdown pages into a single structure (`AggregatedDocument`). This step injects invisible HTML anchors (e.g., `<!-- page_0001 -->`) to preserve physical page boundaries in the continuous text flow.

In [4]:
from doc_agent.data.document_aggregator import aggregate_document
from pathlib import Path

logger.info(f"Starting aggregation for {DOC_ID}...")

# Execute aggregation
agg_doc = aggregate_document(
    doc_id=DOC_ID,
    manifest=manifest,
    storage=storage
)

if agg_doc:
    logger.info(f"Aggregation successful! Total pages merged: {agg_doc.total_pages}")
    
    # Save the full text to a local markdown file for easy visual inspection
    preview_file = Path(f"aggregated_preview_{DOC_ID}.md")
    preview_file.write_text(agg_doc.full_text, encoding="utf-8")
    
    logger.info(f"Saved full aggregated text with HTML anchors to: {preview_file.absolute()}")
    
    # Let's still peek at the terminal output just to be sure
    logger.info("Terminal preview of the top 500 characters:")
    print(agg_doc.full_text[:500] + "\n...\n")
else:
    logger.error("Aggregation failed.")

2026-05-18 17:32:32 |     INFO | 008_aggregation_and_chunking_test:323673187.py:4 - Starting aggregation for pue_1.1-1.3...
2026-05-18 17:32:32 |     INFO | 008_aggregation_and_chunking_test:323673187.py:14 - Aggregation successful! Total pages merged: 27
2026-05-18 17:32:32 |     INFO | 008_aggregation_and_chunking_test:323673187.py:20 - Saved full aggregated text with HTML anchors to: /Volumes/SSD/AI/doc_agent/notebooks/aggregated_preview_pue_1.1-1.3.md
2026-05-18 17:32:32 |     INFO | 008_aggregation_and_chunking_test:323673187.py:23 - Terminal preview of the top 500 characters:
<!-- page_0001 -->
# Раздел 1. ОБЩИЕ ПРАВИЛА

## Глава 1.1 ОБЩАЯ ЧАСТЬ

Утверждено Министерством энергетики Российской Федерации Приказ от 8 июля 2002 г. № 204. Введено в действие с 1 января 2003 г.

### Область применения, определения

#### 1.1.1.
Правила устройства электроустановок (ПУЭ) распространяются на вновь сооружаемые и реконструируемые электроустановки постоянного и переменного тока напряжением до 

## Step 3: Structural Chunking
Now we pass the `AggregatedDocument` to the `StructuralChunker`. This engine uses a State Machine to read the text line-by-line, splitting it strictly at header boundaries (H1, H2, H3, H4) while tracking the HTML page anchors to build rich metadata for the Vector Database.

In [5]:
from doc_agent.indexing.structural_chunker import StructuralChunker
import pandas as pd

logger.info(f"Starting structural chunking for {DOC_ID}...")

# Initialize and execute the chunker
chunker = StructuralChunker()
vector_payloads = chunker.process_document(agg_doc=agg_doc, manifest=manifest.state)

logger.info(f"Chunking complete. Generated {len(vector_payloads)} semantic chunks.")


# Build a summary list to load into a DataFrame
chunk_summaries = []
for p in vector_payloads:
    meta = p.metadata
    chunk_summaries.append({
        "Pages": ", ".join(meta.pages),
        "H1": meta.h1 or "-",
        "H2": meta.h2 or "-",
        "H3": meta.h3 or "-",
        "H4": meta.h4 or "-",
        "Text Snippet": p.text[:80].replace("\n", " ") + "..."
    })

df_chunks = pd.DataFrame(chunk_summaries)

display(Markdown("### Chunks Metadata Overview"))
display(df_chunks.head(15)) # Display the first 15 chunks as a table

2026-05-18 17:32:33 |     INFO | 008_aggregation_and_chunking_test:2006624470.py:4 - Starting structural chunking for pue_1.1-1.3...
2026-05-18 17:32:33 |     INFO | 008_aggregation_and_chunking_test:2006624470.py:10 - Chunking complete. Generated 47 semantic chunks.


### Chunks Metadata Overview

,Pages,H1,H2,H3,H4,Text Snippet
0,page_0001,Раздел 1. ОБЩИЕ ПРАВИЛА,-,-,-,# Раздел 1. ОБЩИЕ ПРАВИЛА...
1,page_0001,Раздел 1. ОБЩИЕ ПРАВИЛА,Глава 1.1 ОБЩАЯ ЧАСТЬ,-,-,## Глава 1.1 ОБЩАЯ ЧАСТЬ Утверждено Министерс...
2,"page_0001, page_0002",Раздел 1. ОБЩИЕ ПРАВИЛА,Глава 1.1 ОБЩАЯ ЧАСТЬ,"Область применения, определения",-,"### Область применения, определения #### 1.1...."
3,"page_0002, page_0003, page_0004",Раздел 1. ОБЩИЕ ПРАВИЛА,Глава 1.1 ОБЩАЯ ЧАСТЬ,Общие указания по устройству электроустановок,-,### Общие указания по устройству электроустано...
4,page_0004,Раздел 1. ОБЩИЕ ПРАВИЛА,Глава 1.1 ОБЩАЯ ЧАСТЬ,Общие указания по устройству электроустановок,1.1.32.,#### 1.1.32. Электроустановки по условиям эле...
5,page_0004,Раздел 1. ОБЩИЕ ПРАВИЛА,Глава 1.1 ОБЩАЯ ЧАСТЬ,Общие указания по устройству электроустановок,1.1.33.,#### 1.1.33. В электропомещениях с установкам...
6,page_0004,Раздел 1. ОБЩИЕ ПРАВИЛА,Глава 1.1 ОБЩАЯ ЧАСТЬ,Общие указания по устройству электроустановок,1.1.34.,"#### 1.1.34. В жилых, общественных и других п..."
7,page_0004,Раздел 1. ОБЩИЕ ПРАВИЛА,Глава 1.1 ОБЩАЯ ЧАСТЬ,Общие указания по устройству электроустановок,1.1.35.,#### 1.1.35. Все ограждающие и закрывающие ус...
8,page_0004,Раздел 1. ОБЩИЕ ПРАВИЛА,Глава 1.1 ОБЩАЯ ЧАСТЬ,Общие указания по устройству электроустановок,1.1.36.,#### 1.1.36. Для защиты обслуживающего персон...
9,page_0004,Раздел 1. ОБЩИЕ ПРАВИЛА,Глава 1.1 ОБЩАЯ ЧАСТЬ,Общие указания по устройству электроустановок,1.1.37.,#### 1.1.37. Пожаро- и взрывобезопасность эле...


## Step 4: Payload Verification (Data Contracts)
Let's inspect the generated `VectorPayload` objects. These are the exact Pydantic models that will be serialized and pushed to **Qdrant**. We need to ensure that the hierarchical metadata (sections, chapters, topic) and the physical page links (PNG artifacts) are perfectly mapped.

In [7]:
from IPython.display import display, Markdown

display(Markdown("### 🔍 Detailed Chunk Inspection"))

# We define backticks in a variable so it doesn't break the Markdown renderer!
md_block = "```"

# Inspect the first 5 chunks to get a clear view of the metadata and text
for i, payload in enumerate(vector_payloads[:5]):
    meta = payload.metadata
    
    # 1. Calculate text length (crucial for embedding models)
    text_len = len(payload.text)
    
    # 2. Clean up the hierarchy display (ignore 'None' values)
    headers = [meta.h1, meta.h2, meta.h3, meta.h4]
    hierarchy_path = " > ".join([h for h in headers if h]) or "*No headers*"
    
    # 3. Format the artifact dictionary nicely
    artifacts = ", ".join([f"{k}: {v}" for k, v in meta.page_artifacts.items()]) or "*None*"

    # 4. Build a beautiful Markdown string safely
    inspection_md = f"""
#### CHUNK {i+1} 
**ID:** `{payload.chunk_id}` | **Length:** {text_len} chars | **Pages:** {', '.join(meta.pages)}

**Hierarchy:** {hierarchy_path}  
**Artifacts:** {artifacts}

**Text Content:**
{md_block}markdown
{payload.text[:600]} ... (truncated)
{md_block}
---
"""
    display(Markdown(inspection_md))

### 🔍 Detailed Chunk Inspection


#### CHUNK 1 
**ID:** `6202daf533284da4` | **Length:** 25 chars | **Pages:** page_0001

**Hierarchy:** Раздел 1. ОБЩИЕ ПРАВИЛА  
**Artifacts:** page_0001: 02_renders_png/page_0001_highres.png

**Text Content:**
```markdown
# Раздел 1. ОБЩИЕ ПРАВИЛА ... (truncated)
```
---



#### CHUNK 2 
**ID:** `c2e1315f400b102c` | **Length:** 152 chars | **Pages:** page_0001

**Hierarchy:** Раздел 1. ОБЩИЕ ПРАВИЛА > Глава 1.1 ОБЩАЯ ЧАСТЬ  
**Artifacts:** page_0001: 02_renders_png/page_0001_highres.png

**Text Content:**
```markdown
## Глава 1.1 ОБЩАЯ ЧАСТЬ

Утверждено Министерством энергетики Российской Федерации Приказ от 8 июля 2002 г. № 204. Введено в действие с 1 января 2003 г. ... (truncated)
```
---



#### CHUNK 3 
**ID:** `e25a8c843e2e5bc1` | **Length:** 6233 chars | **Pages:** page_0001, page_0002

**Hierarchy:** Раздел 1. ОБЩИЕ ПРАВИЛА > Глава 1.1 ОБЩАЯ ЧАСТЬ > Область применения, определения  
**Artifacts:** page_0001: 02_renders_png/page_0001_highres.png, page_0002: 02_renders_png/page_0002_highres.png

**Text Content:**
```markdown
### Область применения, определения

#### 1.1.1.
Правила устройства электроустановок (ПУЭ) распространяются на вновь сооружаемые и реконструируемые электроустановки постоянного и переменного тока напряжением до 750 кВ, в том числе на специальные электроустановки, рассмотренные в разд. 7 настоящих Правил.

Устройство специальных электроустановок, не рассмотренных в разд. 7, должно регламентироваться другими нормативными документами. Отдельные требования настоящих Правил могут применяться для таких электроустановок в той мере, в какой они по исполнению и условиям работы аналогичны электроустанов ... (truncated)
```
---



#### CHUNK 4 
**ID:** `4241ea968c9d20a0` | **Length:** 7741 chars | **Pages:** page_0002, page_0003, page_0004

**Hierarchy:** Раздел 1. ОБЩИЕ ПРАВИЛА > Глава 1.1 ОБЩАЯ ЧАСТЬ > Общие указания по устройству электроустановок  
**Artifacts:** page_0002: 02_renders_png/page_0002_highres.png, page_0003: 02_renders_png/page_0003_highres.png, page_0004: 02_renders_png/page_0004_highres.png

**Text Content:**
```markdown
### Общие указания по устройству электроустановок

#### 1.1.19.
Применяемые в электроустановках электрооборудование, электротехнические изделия и материалы должны соответствовать требованиям государственных стандартов или технических условий, утвержденных в установленном порядке.

#### 1.1.20.
Конструкция, исполнение, способ установки, класс и характеристики изоляции применяемых машин, аппаратов, приборов и прочего электрооборудования, а также кабелей и проводов должны соответствовать параметрам сети или электроустановки, режимам работы, условиям окружающей среды и требованиям соответствующих  ... (truncated)
```
---



#### CHUNK 5 
**ID:** `8a174da32a8603ad` | **Length:** 964 chars | **Pages:** page_0004

**Hierarchy:** Раздел 1. ОБЩИЕ ПРАВИЛА > Глава 1.1 ОБЩАЯ ЧАСТЬ > Общие указания по устройству электроустановок > 1.1.32.  
**Artifacts:** page_0004: 02_renders_png/page_0004_highres.png

**Text Content:**
```markdown
#### 1.1.32.

Электроустановки по условиям электробезопасности разделяются на электроустановки напряжением до 1 кВ и электроустановки напряжением выше 1 кВ (по действующему значению напряжения).

Безопасность обслуживающего персонала и посторонних лиц должна обеспечиваться выполнением мер защиты, предусмотренных в гл. 1.7, а также следующих мероприятий:

- соблюдение соответствующих расстояний до токоведущих частей или путем закрытия, ограждения токоведущих частей;
- применение блокировки аппаратов и ограждающих устройств для предотвращения ошибочных операций и доступа к токоведущим частям;
-  ... (truncated)
```
---


## Conclusion & Next Steps

If the output above successfully displays discrete text blocks tied to specific headers and maps back to the physical `page_XXXX_highres.png` files.

The final `VectorPayload` objects are now ready to be passed to an embedding model and upserted into the Qdrant vector database.